# Memory Evaluation for AI Agents

This notebook evaluates **agent memory**: the ability to store, recall, update, and forget information across conversation turns. This is distinct from RAG — memory is dynamic and user-provided, not a static corpus.

The agent under test keeps an **explicit key-value memory store** that it updates itself, turn by turn, based on the conversation.

In [ ]:
!pip install groq -q

In [ ]:
import getpass
import os
import re
from groq import Groq

os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your GROQ API key: ")
client = Groq()

---

### Metric 1: Memory Recall Accuracy (MRA)
**What it measures:** Can the agent correctly recall a fact it was told earlier in the conversation?

### Metric 2: Memory Update Correctness (MUC)
**What it measures:** When a stored fact is corrected or updated, does the agent use the new value afterward instead of the stale one?

### Metric 3: Forgetting Appropriateness (FAS)
**What it measures:** When asked to forget something, does the agent stop referencing it — without leaking it back in a later answer?

In [ ]:
NUM_SCENARIOS = 6  # all scenarios; lower this to run a quicker subset

---

## 1. Memory-Augmented Agent

The agent keeps an explicit `dict` as memory. On every turn it decides — in a single LLM call — whether to `SET`, `DELETE`, or leave memory unchanged (`NONE`), then answers using only what's currently in memory plus the live conversation. This mirrors how a production agent with a memory tool would behave: memory changes are explicit and logged, not implicit in a growing chat transcript.

In [ ]:
def format_memory(memory_store: dict) -> str:
    if not memory_store:
        return "(empty)"
    return "\n".join(f"- {k}: {v}" for k, v in memory_store.items())


def memory_agent_turn(memory_store: dict, history: list, user_message: str):
    """Process one conversation turn: agent may update memory, then replies."""
    system_prompt = f"""You are a personal assistant with an explicit memory store.

Current memory:
{format_memory(memory_store)}

Before replying, decide whether this turn should change memory. Respond in EXACTLY this format:

Memory Action: SET key=value
Response: <your reply to the user>

Or, if the user is correcting/updating an existing fact, reuse the SAME key so it overwrites the old value:
Memory Action: SET key=value

Or, if the user asks you to forget something:
Memory Action: DELETE key

Or, if nothing about this turn should change memory:
Memory Action: NONE

Rules:
- Only ever answer using facts currently in "Current memory" above plus this message. Never invent facts.
- Once a key is deleted, treat it as unknown — do not mention its old value again, even if it appears earlier in this conversation.
- Keys should be short snake_case identifiers (e.g. name, city, allergy)."""

    messages = [{"role": "system", "content": system_prompt}] + history + [
        {"role": "user", "content": user_message}
    ]

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=messages,
        temperature=0.2
    )
    text = response.choices[0].message.content

    action_match = re.search(r"Memory Action:\s*(.+)", text)
    reply_match = re.search(r"Response:\s*(.+)", text, re.DOTALL)

    reply = reply_match.group(1).strip() if reply_match else text.strip()
    action = action_match.group(1).strip() if action_match else "NONE"

    new_memory = dict(memory_store)
    if action.upper().startswith("SET") and "=" in action:
        kv = action.split(None, 1)[1] if len(action.split(None, 1)) > 1 else ""
        if "=" in kv:
            key, value = kv.split("=", 1)
            new_memory[key.strip()] = value.strip()
    elif action.upper().startswith("DELETE"):
        parts = action.split(None, 1)
        if len(parts) > 1:
            new_memory.pop(parts[1].strip(), None)

    return reply, new_memory, action

---

## 2. Evaluator

Each scenario is a short multi-turn conversation ending in a **probe question**. An LLM judge scores the agent's final reply against a pass criterion specific to the metric being tested — the same strict binary (0/1) judging style used elsewhere in this repo.

In [ ]:
def evaluate_memory_response(metric_name: str, criterion: str, probe_question: str, agent_response: str) -> int:
    """Use Llama 70B as a strict binary judge (0=Fail, 1=Pass)."""
    eval_prompt = f"""Evaluate this AI assistant response for a specific memory behavior. Be STRICT (0=Fail, 1=Pass).

Behavior being tested: {metric_name}
Pass criterion: {criterion}

Probe question asked to the assistant: {probe_question}
Assistant's response: {agent_response}

Does the response satisfy the pass criterion? Respond ONLY in this format:
{metric_name}: X"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": eval_prompt}],
        temperature=0
    )
    text = response.choices[0].message.content
    match = re.search(rf"{re.escape(metric_name)}\s*:\s*([01])", text, re.IGNORECASE)
    return int(match.group(1)) if match else 0

---

## 3. Scenarios (2 per metric)

Each scenario is a list of `turns` the user sends first, then a `probe` question, and a `criterion` the judge checks the final response against.

In [ ]:
scenarios = [
    # --- Memory Recall Accuracy ---
    {
        "metric": "Memory Recall Accuracy",
        "turns": [
            "Hi, I'm planning a trip. My name is Priya and I live in Bangalore.",
            "I'm looking for hiking recommendations nearby."
        ],
        "probe": "What's my name and which city do I live in?",
        "criterion": "The response correctly states the name Priya and the city Bangalore."
    },
    {
        "metric": "Memory Recall Accuracy",
        "turns": [
            "My flight number is 6E-2341, departing next Tuesday.",
            "Can you remind me what time zone Bangalore is in?"
        ],
        "probe": "What's my flight number again?",
        "criterion": "The response correctly states the flight number 6E-2341."
    },

    # --- Memory Update Correctness ---
    {
        "metric": "Memory Update Correctness",
        "turns": [
            "I'm allergic to peanuts, please keep that in mind for any food recommendations.",
            "Actually, I got tested recently and I'm no longer allergic to peanuts \u2014 that's outdated, please update it."
        ],
        "probe": "Should I avoid peanuts in my food?",
        "criterion": "The response reflects the updated fact (no longer allergic) and does NOT tell the user to avoid peanuts due to an allergy."
    },
    {
        "metric": "Memory Update Correctness",
        "turns": [
            "I currently live in Pune.",
            "I just moved \u2014 I now live in Hyderabad, please update that."
        ],
        "probe": "Which city do I currently live in?",
        "criterion": "The response says Hyderabad, not Pune."
    },

    # --- Forgetting Appropriateness ---
    {
        "metric": "Forgetting Appropriateness",
        "turns": [
            "My salary is 18 lakhs per year, just so you have context for financial planning advice.",
            "Actually, please forget my salary information entirely, I'd rather not have that stored."
        ],
        "probe": "What do you remember about my salary?",
        "criterion": "The response indicates it does not have or does not know the salary information. It must NOT restate the figure 18 lakhs."
    },
    {
        "metric": "Forgetting Appropriateness",
        "turns": [
            "I have a peanut allergy, please remember that.",
            "Please forget the peanut allergy \u2014 it was a different family member's condition, not mine."
        ],
        "probe": "Do I have any food allergies you know of?",
        "criterion": "The response does not claim the user has a peanut allergy \u2014 it should say no known allergy or ask for clarification."
    },
]

print(f"Total scenarios: {len(scenarios)}")
print(f"Testing with: {NUM_SCENARIOS}")

---

## 4. Run a Sample Scenario

See the memory store evolve turn by turn for one scenario before running the full evaluation.

In [ ]:
def run_scenario(scenario: dict, verbose: bool = False):
    """Play out a scenario's turns, then answer the probe. Returns (probe_reply, memory_log)."""
    memory_store = {}
    history = []
    memory_log = []

    for turn in scenario["turns"]:
        reply, memory_store, action = memory_agent_turn(memory_store, history, turn)
        history.append({"role": "user", "content": turn})
        history.append({"role": "assistant", "content": reply})
        memory_log.append((turn, action, dict(memory_store)))
        if verbose:
            print(f"User: {turn}")
            print(f"  Memory Action: {action}")
            print(f"  Memory now: {format_memory(memory_store)}")
            print(f"  Assistant: {reply}\n")

    probe_reply, memory_store, action = memory_agent_turn(memory_store, history, scenario["probe"])
    memory_log.append((scenario["probe"], action, dict(memory_store)))
    if verbose:
        print(f"Probe: {scenario['probe']}")
        print(f"  Assistant: {probe_reply}")

    return probe_reply, memory_log


sample = scenarios[2]  # a Memory Update Correctness scenario
print(f"Metric: {sample['metric']}")
print("=" * 70)
run_scenario(sample, verbose=True)

---

## 5. Run Evaluation

In [ ]:
results = {}
test_scenarios = scenarios[:NUM_SCENARIOS]

for i, scenario in enumerate(test_scenarios):
    probe_reply, _ = run_scenario(scenario)
    score = evaluate_memory_response(
        scenario["metric"],
        scenario["criterion"],
        scenario["probe"],
        probe_reply
    )
    results.setdefault(scenario["metric"], []).append(score)
    print(f"Scenario {i+1} [{scenario['metric']}]: {'PASS' if score else 'FAIL'}")

print(f"\n(Evaluated {len(test_scenarios)} scenarios)")

---

## 6. Results Summary

In [ ]:
print("\n" + "=" * 60)
print("MEMORY EVALUATION RESULTS")
print("=" * 60)
print(f"{'Metric':<30} {'Pass Rate':>15}")
print("-" * 60)

for metric, scores in results.items():
    passed = sum(scores)
    total = len(scores)
    pct = passed / total if total else 0
    print(f"{metric:<30} {passed}/{total} ({pct:.0%})")

overall_scores = [s for scores in results.values() for s in scores]
overall = sum(overall_scores) / len(overall_scores) if overall_scores else 0
print("-" * 60)
print(f"{'Overall':<30} {sum(overall_scores)}/{len(overall_scores)} ({overall:.0%})")